In [1]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# -----------------------------
# configuration
# -----------------------------
num_p = 10
num_trials = 100
np.random.seed(1)
# -----------------------------
# Newton's Method for
# x^2 + bx + c = 0
# -----------------------------
def newton_root(b, c, x0, max_iter=50, tol=1e-10):
    x = x0

    for _ in range(max_iter):
        f = x**2 + b*x + c
        df = 2*x + b

        if abs(df) < 1e-12:
            return None

        x_new = x - f / df

        if abs(x_new - x) < tol:
            return x_new

        x = x_new

    return None   # did not converge in max_iter steps

# -----------------------------
# Learn the positive branch
# u = (-b + sqrt(b^2 - 4c)) / 2
# Generate training data using
# Newton's Method
# -----------------------------
b_vec = []
c_vec = []
u_vec = []

while len(b_vec) < num_trials:
    b = np.random.uniform(-10, 10)
    c = np.random.uniform(-10, 10)

    # only use equations with real roots
    if b**2 - 4*c > 0:
        # use Newton's Method
        u = newton_root(b, c, x0=10)

        if u is not None:
            b_vec.append(b)
            c_vec.append(c)
            u_vec.append(u)

b_vec = np.array(b_vec)
c_vec = np.array(c_vec)
u_vec = np.array(u_vec)

In [2]:
def hypothesis(params, b, c):
    xi0, xi1, xi2, xi3, xi4, xi5, xi6, xi7, xi8, xi9 = params

    inside_sqrt = (
        xi4
        + xi5*b
        + xi6*c
        + xi7*b**2
        + xi8*c**2
        + xi9*b*c
    )

    inside_sqrt = np.maximum(inside_sqrt, 1e-8)

    return xi0 + xi1*b + xi2*c + xi3*np.sqrt(inside_sqrt)

In [3]:
def objective(params):
    J = 0.0

    for b, c, u in zip(b_vec, c_vec, u_vec):
        guess = hypothesis(params, b, c)
        J += 0.5 * (guess - u)**2

    return J / len(u_vec)

In [4]:
best_result = None
best_error = np.inf

for trial in range(20):
    x0 = np.random.randn(num_p)

    result = minimize(
        objective,
        x0,
        method="L-BFGS-B",
        options={
            "maxiter": 5000,
            "ftol": 1e-10
        }
    )

    error = objective(result.x)

    if error < best_error:
        best_error = error
        best_result = result

learned_params = best_result.x

print("Best training error:")
print(best_error)

print("\nLearned parameters")
print(learned_params)

Best training error:
3.1921802427642024e-10

Learned parameters
[-6.46322506e-05 -4.99995523e-01 -1.16367406e-06  4.29629706e-01
  6.71895167e-04 -1.11410545e-04 -5.41781788e+00  1.35446480e+00
 -1.13907305e-05  3.65756978e-06]


In [6]:
# Normalize after optimization to remove scaling ambiguity in the square-root term.
def normalize_params(params):
    params = params.copy()

    xi7 = params[7]  # coefficient of b^2 inside sqrt

    if xi7 != 0:
        s = np.sign(xi7)
        params[3] = params[3] * np.sqrt(xi7)  # adjust outside sqrt coefficient
        params[4:10] = params[4:10] / xi7     # divide inside sqrt coefficients
        params[7] = s                      # force b^2 coefficient to +1 or -1

    return params

learned_params = normalize_params(learned_params)

print("Normalized learned parameters:")
print(learned_params)

Normalized learned parameters:
[-6.46322506e-05 -4.99995523e-01 -1.16367406e-06  5.00009394e-01
  4.96059527e-04 -8.22542937e-05 -3.99996950e+00  1.00000000e+00
 -8.40976486e-06  2.70038009e-06]


In [7]:
threshold = np.max(np.abs(learned_params)) / 10

sparse_params = learned_params.copy()
sparse_params[np.abs(sparse_params) < threshold] = 0

print("Threshold:")
print(threshold)

print("\nSparse parameters:")
print(sparse_params)

Threshold:
0.39999694968018057

Sparse parameters:
[ 0.         -0.49999552  0.          0.50000939  0.          0.
 -3.9999695   1.          0.          0.        ]


In [8]:
# -----------------------------
# Learn the negative branch
# v = (-b - sqrt(b^2 - 4c)) / 2
# Generate training data using
# Newton's Method
# -----------------------------
b_vec = []
c_vec = []
v_vec = []

while len(b_vec) < num_trials:
    b = np.random.uniform(-10, 10)
    c = np.random.uniform(-10, 10)

    # only use equations with real roots
    if b**2 - 4*c > 0:
        # use Newton's Method
        v = newton_root(b, c, x0=-10)

        if v is not None:
            b_vec.append(b)
            c_vec.append(c)
            v_vec.append(v)

b_vec = np.array(b_vec)
c_vec = np.array(c_vec)
v_vec = np.array(v_vec)


In [9]:
def objective(params):
    J = 0.0

    for b, c, v in zip(b_vec, c_vec, v_vec):
        guess = hypothesis(params, b, c)
        J += 0.5 * (guess - v)**2

    return J / len(v_vec)

In [10]:
best_result = None
best_error = np.inf

for trial in range(20):
    x0 = np.random.randn(num_p)

    result = minimize(
        objective,
        x0,
        method="L-BFGS-B",
        options={
            "maxiter": 5000,
            "ftol": 1e-10
        }
    )

    error = objective(result.x)

    if error < best_error:
        best_error = error
        best_result = result

learned_params = best_result.x



learned_params = normalize_params(learned_params)

print("Normalized learned parameters:")
print(learned_params)

print("\nTraining error:")
print(objective(learned_params))
threshold = np.max(np.abs(learned_params)) / 10

sparse_params = learned_params.copy()
sparse_params[np.abs(sparse_params) < threshold] = 0

print("Threshold:")
print(threshold)

print("\nSparse parameters:")
print(sparse_params)

Normalized learned parameters:
[ 3.44718688e-05 -4.99999979e-01 -9.86356983e-07 -5.00004515e-01
  1.01609143e-04  8.75962294e-07 -4.00000867e+00  1.00000000e+00
 -1.65530798e-06 -4.61316679e-06]

Training error:
4.931656961576144e-11
Threshold:
0.40000086652466377

Sparse parameters:
[ 0.         -0.49999998  0.         -0.50000451  0.          0.
 -4.00000867  1.          0.          0.        ]
